# Enrichissement financier — CA & Résultat net via API INPI

Ce notebook récupère les leads depuis Supabase, interroge l'API INPI (comptes annuels déposés aux greffes)
pour extraire le chiffre d'affaires et le résultat net, puis met à jour Supabase.

## 1. Configuration

In [ ]:
# ─── INPI API (créer un compte sur https://data.inpi.fr) ───
INPI_USERNAME = "maxime.debaugnies@cloudkitchens.com"
INPI_PASSWORD = "Thomas36130!"
INPI_BASE_URL = "https://registre-national-entreprises.inpi.fr/api"

# ─── Supabase ───
SUPABASE_URL = "https://hxjryfaakdpwfgseirik.supabase.co"
SUPABASE_API_KEY = "sb_publishable_a7xpn8srwByQrW1rXwpdlw_eQnwAoHT"
SUPABASE_TABLE = "leads"

# ─── Paramètres ───
BATCH_LIMIT = 500     # Nombre de leads à traiter par exécution (~1000 req API)
REQUEST_DELAY = 0.5   # Pause entre chaque lead (secondes)

## 1b. Colonnes Supabase à créer (une seule fois)

Copie ce SQL dans le **SQL Editor** du dashboard Supabase :

```sql
ALTER TABLE leads
  ADD COLUMN resultat_net bigint,
  ADD COLUMN date_cloture_bilan text,
  ADD COLUMN enriched_inpi boolean DEFAULT false;
```

## 2. Imports

In [ ]:
import requests
import time
import json

## 3. Authentification INPI

In [ ]:
def inpi_login(username, password):
    """Obtient un token Bearer via le SSO INPI."""
    resp = requests.post(
        f"{INPI_BASE_URL}/sso/login",
        json={"username": username, "password": password},
        headers={"Content-Type": "application/json"},
    )
    if resp.status_code == 200:
        token = resp.json()["token"]
        print(f"✓ Connecté à l'API INPI")
        return token
    else:
        print(f"✗ Erreur auth INPI : HTTP {resp.status_code} — {resp.text[:300]}")
        raise RuntimeError("Authentification INPI échouée.")


token = inpi_login(INPI_USERNAME, INPI_PASSWORD)
INPI_HEADERS = {"Authorization": f"Bearer {token}"}

## 4. Chargement des leads depuis Supabase

In [ ]:
def fetch_leads_to_enrich(limit):
    """Charge les leads sans CA depuis Supabase."""
    resp = requests.get(
        f"{SUPABASE_URL}/rest/v1/{SUPABASE_TABLE}",
        params={
            "select": "siren,nom,ville",
            "chiffre_affaires": "is.null",
            "enriched_inpi": "eq.false",
            "limit": limit,
        },
        headers={
            "apikey": SUPABASE_API_KEY,
            "Authorization": f"Bearer {SUPABASE_API_KEY}",
            "Accept": "application/json",
        },
    )
    resp.raise_for_status()
    leads = resp.json()
    print(f"✓ {len(leads)} leads à enrichir")
    return leads


leads = fetch_leads_to_enrich(BATCH_LIMIT)

## 5. Exploration — Inspecter la réponse INPI (à lancer une fois)

Cette cellule teste l'API sur un SIREN pour voir la structure exacte de la réponse.
Après validation, passer directement à la section 6.

In [ ]:
TEST_SIREN = leads[0]["siren"] if leads else "552032534"

resp_attach = requests.get(
    f"{INPI_BASE_URL}/companies/{TEST_SIREN}/attachments",
    headers=INPI_HEADERS,
)
print(f"=== Attachments HTTP {resp_attach.status_code} ===")
data = resp_attach.json()
print(f"actes: {len(data.get('actes', []))}, bilans: {len(data.get('bilans', []))}, bilansSaisis: {len(data.get('bilansSaisis', []))}")

bs = data.get("bilansSaisis", [])
if bs:
    b = bs[0]
    identite = b["bilanSaisi"]["bilan"]["identite"]
    print(f"\nDernier bilan : type={identite['codeTypeBilan']}, clôture={identite['dateClotureExercice']}")
    print(f"Confidentiel : {b['confidentiality']}")

    for page in b["bilanSaisi"]["bilan"]["detail"]["pages"]:
        for liasse in page["liasses"]:
            if liasse["code"] in ("FJ", "FL", "HN"):
                print(f"  {liasse['code']}: {json.dumps(liasse)}")
else:
    print("Aucun bilan saisi pour ce SIREN")

## 6. Enrichissement INPI

In [ ]:
def parse_montant(val):
    """Parse un montant INPI (string zéro-paddée, potentiellement négative) en entier."""
    if not val:
        return None
    val = val.strip()
    try:
        return int(val)
    except (ValueError, TypeError):
        return None


def extract_financial_data(bilan_saisi):
    """Extrait CA et résultat net depuis un bilanSaisi inclus dans attachments."""
    bilan = bilan_saisi.get("bilanSaisi", {}).get("bilan", {})
    identite = bilan.get("identite", {})
    date_cloture = identite.get("dateClotureExercice", "")

    ca = None
    resultat_net = None

    for page in bilan.get("detail", {}).get("pages", []):
        for liasse in page.get("liasses", []):
            code = liasse.get("code", "")

            if code == "FL":  # CA net total (certains types de bilan)
                ca = parse_montant(liasse.get("m3") or liasse.get("m1"))
            elif code == "FJ" and ca is None:  # Ventes de marchandises (fallback)
                ca = parse_montant(liasse.get("m3") or liasse.get("m1"))
            elif code == "HN":  # Résultat net
                resultat_net = parse_montant(liasse.get("m1"))

    if ca is None and resultat_net is None:
        return None

    return {
        "chiffre_affaires": ca,
        "resultat_net": resultat_net,
        "date_cloture_bilan": date_cloture,
    }


def get_financial_data(siren):
    """Récupère les données financières d'un SIREN via l'API INPI (1 seul appel)."""
    resp = requests.get(
        f"{INPI_BASE_URL}/companies/{siren}/attachments",
        headers=INPI_HEADERS,
    )
    if resp.status_code == 404:
        return None
    if resp.status_code == 429:
        print("⏳ rate limit, pause 30s… ", end="")
        time.sleep(30)
        return get_financial_data(siren)
    if resp.status_code != 200:
        return None

    bilans_saisis = resp.json().get("bilansSaisis", [])
    for bs in bilans_saisis:
        if bs.get("confidentiality") == "Public" and not bs.get("deleted", False):
            return extract_financial_data(bs)
    return None


enriched = []
found_ca = 0
no_bilan = 0
errors = 0

for i, lead in enumerate(leads):
    siren = lead["siren"]
    nom = lead["nom"]
    print(f"[{i+1}/{len(leads)}] {nom[:50]:50s} ", end="")

    financial = get_financial_data(siren)
    if financial is None:
        print("— ✗ pas de bilan")
        no_bilan += 1
        enriched.append({"siren": siren, "data": None})
        time.sleep(REQUEST_DELAY)
        continue

    ca = financial["chiffre_affaires"]
    rn = financial["resultat_net"]
    parts = []
    if ca is not None:
        parts.append(f"CA={ca:,}€")
        found_ca += 1
    if rn is not None:
        parts.append(f"RN={rn:,}€")
    if parts:
        print(f"— ✓ {' | '.join(parts)}")
    else:
        print("— ✗ données vides")
        errors += 1

    enriched.append({"siren": siren, "data": financial})
    time.sleep(REQUEST_DELAY)

print(f"\n══ Résultat : {found_ca}/{len(leads)} avec CA, {no_bilan} sans bilan, {errors} erreurs ══")

## 7. Mise à jour Supabase

In [ ]:
updated = 0
skipped = 0

for entry in enriched:
    siren = entry["siren"]
    patch_data = {"enriched_inpi": True}

    if entry["data"]:
        if entry["data"]["chiffre_affaires"] is not None:
            patch_data["chiffre_affaires"] = entry["data"]["chiffre_affaires"]
        if entry["data"]["resultat_net"] is not None:
            patch_data["resultat_net"] = entry["data"]["resultat_net"]
        if entry["data"]["date_cloture_bilan"]:
            patch_data["date_cloture_bilan"] = entry["data"]["date_cloture_bilan"]

    resp = requests.patch(
        f"{SUPABASE_URL}/rest/v1/{SUPABASE_TABLE}?siren=eq.{siren}",
        headers={
            "apikey": SUPABASE_API_KEY,
            "Authorization": f"Bearer {SUPABASE_API_KEY}",
            "Content-Type": "application/json",
            "Prefer": "return=minimal",
        },
        json=patch_data,
    )

    if resp.status_code in (200, 204):
        if len(patch_data) > 1:  # Plus que juste enriched_inpi
            updated += 1
        else:
            skipped += 1
    else:
        print(f"  ✗ SIREN {siren} : HTTP {resp.status_code} — {resp.text[:200]}")

print(f"\n══ {updated} leads enrichis, {skipped} sans données financières ══")